# 25 — Pull V-Party (Varieties of Party Identity and Organization)

**Source:** V-Party Dataset  
**Provider:** V-Dem Institute, University of Gothenburg  
**Access:** `vdemdata` Python package (`load_vparty()`) — free, no registration  
**Coverage:** ~170 countries, major parties, 1970–present  
**Frequency:** Annual (party-year level)  

## What this notebook does
Loads the V-Party dataset and aggregates from **party-year** to **country-year**,
focusing on populist and anti-establishment indicators.

Populist party presence in government is one of the strongest documented predictors
of regime backsliding (Lührmann & Lindberg 2019; Medzihorsky et al. 2021). The main
V-Dem country-level dataset does not capture this; V-Party is the source.

## Aggregation logic
For each country-year we compute:
- **Governing coalition**: max/mean populism score among parties in government  
- **All parties**: max populism among any party above 5% vote share  
- **Binary flags**: whether any governing party exceeds a populism threshold  

## Features produced

| Feature | Description |
|---|---|
| `gov_populism_max` | Max populism score among governing parties |
| `gov_populism_mean` | Mean populism score among governing parties |
| `gov_antiestab_max` | Max anti-establishment score among governing parties |
| `gov_party_violence` | Any governing party coded as using violence |
| `any_populist_gov` | Binary: any governing party with populism ≥ 2 (0–4 scale) |
| `opp_populism_max` | Max populism among opposition parties (>5% vote share) |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import numpy as np
import pandas as pd
import vdemdata
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# V-Party populism threshold for binary flag (scale 0–4)
POPULISM_THRESHOLD = 2.0

print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/vparty/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Load V-Party via vdemdata

In [ ]:
print("Loading V-Party dataset...")
try:
    df_vp_full = vdemdata.load_vparty()
    print(f"V-Party shape: {df_vp_full.shape}")
    print(f"Columns sample: {list(df_vp_full.columns[:20])}")
except AttributeError:
    # Older vdemdata versions may not include load_vparty — fall back to direct download
    print("vdemdata.load_vparty() not available — downloading directly from V-Dem")
    import requests, io
    VPARTY_URL = "https://v-dem.net/media/datasets/V-Party_Dataset_V2.csv.zip"
    resp = requests.get(VPARTY_URL, timeout=120)
    resp.raise_for_status()
    import zipfile
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    csv_name = [n for n in zf.namelist() if n.endswith(".csv")][0]
    df_vp_full = pd.read_csv(zf.open(csv_name), low_memory=False)
    print(f"V-Party shape (direct): {df_vp_full.shape}")

## Select and rename key columns

V-Party variable names follow the v2pa* pattern. We map to descriptive names and
handle version differences gracefully.

In [ ]:
# Core identifiers
ID_COLS = {
    "country_name":       "country_name",
    "country_text_id":    "iso3",
    "country_id":         "country_id",
    "year":               "year",
    "v2pashname":         "party_name",
}

# Key political variables (v2pa* prefix)
VPARTY_VARS = {
    "v2pagovm":    "gov_status",         # 0=opposition, 1=junior, 2=senior governing
    "v2papolit":   "populism",           # Populist rhetoric (0–4)
    "v2paantiquo": "anti_estab",         # Anti-establishment stance (0–4)
    "v2paviol":    "party_violence",     # Violence used by party (0–4)
    "v2paclient":  "clientelism",        # Clientelism (0–4)
    "v2pashame":   "norm_compliance",    # Follows rules/norms (0–4)
    "v2paseatshare":"seat_share",        # Legislature seat share (%)
    "v2pavoteshare":"vote_share",        # Vote share at last election (%)
}

ALL_WANTED = {**ID_COLS, **VPARTY_VARS}
available = {k: v for k, v in ALL_WANTED.items() if k in df_vp_full.columns}
missing   = [k for k in ALL_WANTED if k not in df_vp_full.columns]
if missing:
    print(f"Note: {len(missing)} columns absent from this V-Party release: {missing}")

df_vp = df_vp_full[list(available.keys())].rename(columns=available).copy()
df_vp["year"] = pd.to_numeric(df_vp["year"], errors="coerce").astype("Int64")

print(f"Filtered shape: {df_vp.shape}")
print(f"Countries: {df_vp['country_name'].nunique()} | Years: {df_vp['year'].min()}–{df_vp['year'].max()}")
df_vp.head()

## Aggregate to country-year

Separate governing parties (gov_status ≥ 1) from opposition parties.
Compute max and mean populism for governing coalition; max for opposition.

In [ ]:
for col in ["populism", "anti_estab", "party_violence", "vote_share", "seat_share"]:
    if col in df_vp.columns:
        df_vp[col] = pd.to_numeric(df_vp[col], errors="coerce")

# Governing parties
df_gov = df_vp[df_vp.get("gov_status", pd.Series(dtype=float)).fillna(0) >= 1].copy() \
    if "gov_status" in df_vp.columns else df_vp[df_vp["gov_status"].fillna(0) >= 1].copy()

df_gov_agg = df_gov.groupby(["country_name", "iso3", "year"], as_index=False).agg(
    gov_populism_max  =("populism",      "max"),
    gov_populism_mean =("populism",      "mean"),
    gov_antiestab_max =("anti_estab",    "max"),
    gov_party_violence=("party_violence","max"),
)
df_gov_agg["any_populist_gov"] = (df_gov_agg["gov_populism_max"] >= POPULISM_THRESHOLD).astype(int)

# Opposition parties (vote_share > 5%)
if "vote_share" in df_vp.columns and "gov_status" in df_vp.columns:
    df_opp = df_vp[
        (df_vp["gov_status"].fillna(0) == 0) &
        (df_vp["vote_share"].fillna(0) > 5)
    ]
    df_opp_agg = df_opp.groupby(["country_name", "iso3", "year"], as_index=False).agg(
        opp_populism_max=("populism", "max")
    )
    df_vparty = df_gov_agg.merge(df_opp_agg, on=["country_name", "iso3", "year"], how="left")
else:
    df_vparty = df_gov_agg
    df_vparty["opp_populism_max"] = np.nan

df_vparty = df_vparty.sort_values(["iso3", "year"]).reset_index(drop=True)
print(f"Country-year panel shape: {df_vparty.shape}")
df_vparty.head()

## Write to ADLS

In [ ]:
write_parquet(df_vparty, f"raw/vparty/{RUN_DATE}/vparty_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("V-Party pull complete")
print("=" * 55)
print(f"  Rows (country-years) : {len(df_vparty):,}")
print(f"  Countries            : {df_vparty['iso3'].nunique()}")
print(f"  Year range           : {df_vparty['year'].min()}–{df_vparty['year'].max()}")
pct_populist = df_vparty["any_populist_gov"].mean() * 100
print(f"  Country-years with populist governing party: {pct_populist:.1f}%")
print()
print("ADLS path written:")
print(f"  raw/vparty/{RUN_DATE}/vparty_panel.parquet")